# Unified Evaluation: Accuracy + Performance

Run **multi-benchmark Korean MCQ evaluations** and **GuideLLM performance benchmarks** in unified MLflow experiments.
For single-benchmark evaluation, see **2_eval_hub_kmcq_benchmark/1_kmcq_benchmark.ipynb**.


This notebook runs LLM evaluations through the [EvalHub](https://github.com/eval-hub/eval-hub) REST API using the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) Python client. All results are automatically tracked in **MLflow**.

## Why use EvalHub?

| Feature | LMEvalJob (direct) | EvalHub (Phase 2) |
|---------|----------------------|-------------------|
| Interface | Kubernetes CR (YAML) | Python SDK / REST API |
| Frameworks | lm-evaluation-harness only | lm-eval, RAGAS, LightEval, GuideLLM, ... |
| Multi-benchmark | One task per CR | Multiple benchmarks per request |
| Experiment tracking | Manual (Pod logs) | **Built-in MLflow** (metrics, params, artifacts) |
| Result management | `oc get lmevaljob` | Centralized API + MLflow UI |
| Job management | `oc delete lmevaljob` | SDK `client.jobs.cancel()` |

## Prerequisites

- **0_setup/0_model_deploy.ipynb** completed (model deployed)
- **0_setup/1_LMEval_setup.ipynb** completed (RBAC and secrets)
- **0_setup/2_eval_hub_setup.ipynb** completed (EvalHub SDK installed and verified)
- EvalHub service running on the cluster
- MLflow tracking server accessible from EvalHub

## Step 1: Configuration

In [1]:
import os, subprocess
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "vllm-gemma4-e2b")
BASE_URL = os.getenv("BASE_URL", f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1")
import sys; sys.path.insert(0, '..')
from utils.port_forward import resolve_evalhub_url
EVALHUB_URL = resolve_evalhub_url(namespace=NAMESPACE)
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", "")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
LIMIT = int(os.getenv("LIMIT", "5"))

if ".apps." in MLFLOW_TRACKING_URI:
    os.environ["MLFLOW_TRACKING_TOKEN"] = EVALHUB_AUTH_TOKEN or ""
    os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
    os.environ["MLFLOW_HTTP_REQUEST_HEADER_X-Mlflow-Workspace"] = NAMESPACE

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Endpoint:  {BASE_URL}")
print(f"EvalHub URL:     {EVALHUB_URL}")
print(f"MLflow URI:      {MLFLOW_TRACKING_URI}")
print(f"Sample Limit:    {LIMIT}")

EvalHub already reachable at localhost:8443
Namespace:       demo
Model Name:      qwen36-27b
Model Endpoint:  https://qwen36-27b-kserve-workload-svc.demo.svc.cluster.local:8000/v1/completions
EvalHub URL:     https://localhost:8443
MLflow URI:      https://mlflow.redhat-ods-applications.svc:8443
Sample Limit:    10000


## Step 2: Initialize the EvalHub Client

In [2]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target:             {model.name} @ {model.url}")

TLS verification disabled - skipping CA bundle detection


TLS verification disabled (insecure mode)


EvalHub client connected: https://localhost:8443
Model target:             qwen36-27b @ https://qwen36-27b-kserve-workload-svc.demo.svc.cluster.local:8000/v1/completions


## Step 3: Discover Available Korean Benchmarks

Query EvalHub for benchmarks available through the `lm_evaluation_harness` provider and filter for Korean tasks.

In [3]:
KOREAN_PROVIDER_NAME = "Korean MCQ Evaluation"
KOREAN_PROVIDER_ID = None

for p in client.providers.list():
    if p.name == KOREAN_PROVIDER_NAME:
        KOREAN_PROVIDER_ID = p.resource.id
        print(f"Korean MCQ provider found (id={KOREAN_PROVIDER_ID})")
        break

if not KOREAN_PROVIDER_ID:
    print("ERROR: Korean MCQ provider is not registered.")
    print("  → Cluster owner must run 0_setup/2_eval_hub_setup.ipynb Step A-7 or Part B first.")

all_benchmarks = client.benchmarks.list()

korean_keywords = ["kmmlu", "kobest", "haerae", "klue", "korean", "ko_", "click", "hrm8k"]
korean_benchmarks = [
    bm for bm in all_benchmarks
    if any(kw in bm.id.lower() for kw in korean_keywords)
]

print(f"Total benchmarks available: {len(all_benchmarks)}")
print(f"Korean benchmarks found:    {len(korean_benchmarks)}")
print("=" * 70)
for bm in korean_benchmarks:
    metrics_str = ", ".join(bm.metrics[:3]) if bm.metrics else "N/A"
    print(f"  {bm.id:40s}  metrics=[{metrics_str}]")

Korean MCQ provider already registered (id=korean_mcq)


Total benchmarks available: 210
Korean benchmarks found:    5
  click                                     metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  haerae                                    metrics=[overall_accuracy, category_accuracy]
  kmmlu                                     metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  kmmlu_hard                                metrics=[overall_accuracy, category_accuracy, supercategory_accuracy]
  kobest_boolq                              metrics=[overall_accuracy]


---

## Evaluation 1: Multi-Benchmark Korean Evaluation

Submit multiple Korean benchmarks in a single request. EvalHub orchestrates them concurrently and tracks all results under one MLflow experiment.

In [4]:
import time

TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
    JobStatus.PARTIALLY_FAILED,
}


def wait_for_job(client, job_id, poll_interval=10, max_wait=600):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.effective_state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.state.value}" for b in status.status.benchmarks]
            bm_info = f" | benchmarks: {', '.join(bm_states)}"

        print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    return status


def display_job_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        return

    print("Evaluation Results")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:30s} = {value:.4f}")
                else:
                    print(f"    {name:30s} = {value}")
        else:
            print("  Metrics: (none)")

print("Helper functions defined: wait_for_job, display_job_results")

Helper functions defined: wait_for_job, display_job_results


In [5]:
multi_request = JobSubmissionRequest(
    name="korean-multi-benchmark",
    description="Comprehensive Korean LLM evaluation: KMMLU + CLIcK + HAE-RAE",
    tags=["korean", "comprehensive", "multi-benchmark"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="kmmlu",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT},
        ),
        BenchmarkConfig(
            id="click",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT},
        ),
        BenchmarkConfig(
            id="haerae",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-comprehensive-eval",
        tags=[
            ExperimentTag(key="language", value="korean"),
            ExperimentTag(key="evaluation_type", value="comprehensive"),
        ],
    ),
)

print("Multi-Benchmark Request:")
print(f"  Name:       {multi_request.name}")
print(f"  Model:      {multi_request.model.name}")
print(f"  Benchmarks: {[b.id for b in multi_request.benchmarks]}")
print(f"  Experiment: {multi_request.experiment.name}")
print(f"\nUncomment the next cell to submit.")

Multi-Benchmark Request:
  Name:       korean-multi-benchmark
  Model:      qwen36-27b
  Benchmarks: ['kmmlu', 'click', 'haerae']
  Experiment: korean-comprehensive-eval

Uncomment the next cell to submit.


In [6]:
# SKIPPED: Use Unified Evaluation (Phase A) below instead
print("Skipped - see Phase A below")

Skipped - see Phase A below


---

## Evaluation 2: Sample Size Comparison

Compare evaluation results with different sample limits on the same benchmark. Each evaluation is tracked as a separate MLflow run within the same experiment for easy comparison.

In [7]:
limit_configs = [
    {"name": "kmmlu-limit-100", "limit": 100},
    {"name": "kmmlu-limit-2000", "limit": 2000},
]

limit_jobs = []
for config in limit_configs:
    request = JobSubmissionRequest(
        name=config["name"],
        description=f"KMMLU evaluation with limit={config['limit']}",
        tags=["korean", "kmmlu", "limit-comparison"],
        model=model,
        benchmarks=[
            BenchmarkConfig(
                id="kmmlu",
                provider_id=KOREAN_PROVIDER_ID,
                parameters={
                    "temperature": 0.0,
                    "max_tokens": 16,
                    "limit": config["limit"],
                },
            ),
        ],
        experiment=ExperimentConfig(
            name="kmmlu-limit-comparison",
            tags=[
                ExperimentTag(key="comparison_type", value="sample-size"),
                ExperimentTag(key="limit", value=str(config["limit"])),
            ],
        ),
    )
    limit_jobs.append(request)
    print(f"Prepared: {config['name']} (limit={config['limit']})")

print(f"\n{len(limit_jobs)} jobs ready. Uncomment below to submit.")

Prepared: kmmlu-limit-100 (limit=100)
Prepared: kmmlu-limit-2000 (limit=2000)

2 jobs ready. Uncomment below to submit.


In [8]:
# SKIPPED: Use Unified Evaluation (Phase A) below instead
print("Skipped - see Phase A below")

Skipped - see Phase A below


---

## Evaluation 3: Unified Evaluation (Accuracy + Performance)

Run **Korean MCQ accuracy benchmarks** and **GuideLLM performance benchmarks** together, tracked under a single MLflow experiment. Accuracy jobs run first to avoid load interference with performance measurements.

In [9]:
from datetime import datetime

ts = datetime.now().strftime("%m%d-%H%M")
EXPERIMENT_NAME = f"{MODEL_NAME}-full-eval-{ts}"

# GuideLLM needs the base /v1 endpoint
GUIDELLM_URL = BASE_URL.replace("/v1/completions", "/v1").replace("/v1/chat/completions", "/v1")
guidellm_model = ModelConfig(url=GUIDELLM_URL, name=MODEL_NAME)

print(f"Unified experiment: {EXPERIMENT_NAME}")
print(f"Korean MCQ URL:     {BASE_URL}")
print(f"GuideLLM URL:       {GUIDELLM_URL}")

# --- Phase A: Korean MCQ Accuracy (5 benchmarks) ---
accuracy_benchmarks = ["click", "haerae", "kmmlu", "kmmlu_hard", "kobest_boolq"]
accuracy_jobs = []

for bm in accuracy_benchmarks:
    req = JobSubmissionRequest(
        name=f"unified-{bm}-{ts}",
        description=f"Korean MCQ accuracy: {bm}",
        tags=["unified", "accuracy", MODEL_NAME],
        model=model,
        benchmarks=[
            BenchmarkConfig(
                id=bm,
                provider_id=KOREAN_PROVIDER_ID,
                parameters={"limit": 10000, "tokenizer": os.getenv("TOKENIZER", "")},
            )
        ],
        experiment=ExperimentConfig(name=EXPERIMENT_NAME),
    )
    job = client.jobs.submit(req)
    accuracy_jobs.append((bm, job.id))
    print(f"  [{bm}] submitted: {job.id}")

print(f"\n{len(accuracy_jobs)} accuracy jobs submitted.")

Unified experiment: qwen36-27b-full-eval-0617-1428
Korean MCQ URL:     https://qwen36-27b-kserve-workload-svc.demo.svc.cluster.local:8000/v1/completions
GuideLLM URL:       https://qwen36-27b-kserve-workload-svc.demo.svc.cluster.local:8000/v1


  [click] submitted: ad5af61a-edee-4e2c-8fcf-ec4ae08939cf


  [haerae] submitted: 9e4dfa43-c3d1-48f2-8701-d76b7c14590a


  [kmmlu] submitted: 9638a01c-2342-4017-9808-9ea355edb6f2


  [kmmlu_hard] submitted: 9da5b370-4492-4252-b9a0-7d6b05bb4578


  [kobest_boolq] submitted: 28b4c90f-3cd2-486d-adb6-a0f917f5eb79

5 accuracy jobs submitted.


In [10]:
# Wait for all accuracy jobs to complete
import time

print("Waiting for accuracy jobs...")
for i in range(120):
    states = {}
    for bm, jid in accuracy_jobs:
        j = client.jobs.get(jid)
        states[bm] = j.state.value
    
    done = all(s in ("completed", "failed", "error") for s in states.values())
    status_str = " | ".join(f"{k}={v}" for k, v in states.items())
    print(f"  [{i*15}s] {status_str}")
    
    if done:
        print("\nAll accuracy jobs finished!")
        break
    time.sleep(15)

# Display accuracy results
for bm, jid in accuracy_jobs:
    j = client.jobs.get(jid)
    if hasattr(j, "results") and j.results:
        for r in j.results.benchmarks:
            acc = r.metrics.get("overall_accuracy", "N/A")
            print(f"  {bm:15s} | accuracy={acc}% | mlflow_run={r.mlflow_run_id}")

Waiting for accuracy jobs...


  [0s] click=pending | haerae=pending | kmmlu=pending | kmmlu_hard=pending | kobest_boolq=pending


  [15s] click=running | haerae=running | kmmlu=running | kmmlu_hard=running | kobest_boolq=running


  [30s] click=running | haerae=running | kmmlu=running | kmmlu_hard=running | kobest_boolq=running


  [45s] click=running | haerae=running | kmmlu=running | kmmlu_hard=running | kobest_boolq=running


...

  click           | accuracy=75.9% | mlflow_run=66055426e4974fc6a479cc53f2199ba2


  haerae          | accuracy=16% | mlflow_run=58512d1684b5463aae7c1b8287549cde


  kmmlu           | accuracy=62.5% | mlflow_run=1adfde119df94038a834b92a1ec8fa04


  kmmlu_hard      | accuracy=43.06% | mlflow_run=b7f5c115dd7e474ebea648ea51e5d93d


  kobest_boolq    | accuracy=96.65% | mlflow_run=dfa8c3d0008e40c0aa649b8a963b78d4


### Phase B: GuideLLM Performance

After accuracy jobs complete, run a GuideLLM performance test against the same model. Results are tracked in the **same MLflow experiment** for unified visibility.

In [11]:
# --- Phase B: GuideLLM Performance ---
# Use 'throughput' profile to discover max throughput without specifying a fixed rate.
# This avoids "Invalid rates in sweep" errors with large models (e.g. 32B+).
perf_request = JobSubmissionRequest(
    name=f"unified-perf-{MODEL_NAME}-{ts}",
    description=f"GuideLLM throughput test for {MODEL_NAME}",
    tags=["unified", "performance", "guidellm", MODEL_NAME],
    model=guidellm_model,
    benchmarks=[
        BenchmarkConfig(
            id="throughput",
            provider_id="guidellm",
            parameters={
                "max_seconds": 180,
                "max_requests": 50,
                "data": "prompt_tokens=128,output_tokens=64",
                "request_type": "chat_completions",
            },
        )
    ],
    experiment=ExperimentConfig(name=EXPERIMENT_NAME),
)

perf_job = client.jobs.submit(perf_request)
print(f"GuideLLM job submitted: {perf_job.id}")
print(f"Experiment: {EXPERIMENT_NAME}")

# Wait for performance job
for i in range(60):
    j = client.jobs.get(perf_job.id)
    state = j.state.value
    if state in ("completed", "failed", "error"):
        print(f"\nGuideLLM job {state}!")
        break
    print(f"  [{i*10}s] {state}", end="\r")
    time.sleep(10)

if hasattr(j, "results") and j.results:
    for bm in j.results.benchmarks:
        print(f"\nPerformance metrics (MLflow run: {bm.mlflow_run_id}):")
        for k, v in sorted(bm.metrics.items()):
            print(f"  {k}: {v}")

print(f"\n--- Unified evaluation complete ---")
print(f"All results tracked in MLflow experiment: {EXPERIMENT_NAME}")

GuideLLM job submitted: 15d4b0b0-860d-4f6a-91ba-742aea659e2a
Experiment: qwen36-27b-full-eval-0617-1428



GuideLLM job completed!

Performance metrics (MLflow run: None):
  mean_itl_ms: 85.45799700327365
  mean_ttft_ms: 235.4049610369133
  output_tokens_per_second: 11.402229222897363
  prompt_tokens_per_second: 25.407045203068346
  requests_per_second: 0.17777777777777778

--- Unified evaluation complete ---
All results tracked in MLflow experiment: qwen36-27b-full-eval-0617-1428


### Log GuideLLM Metrics to MLflow (Manual)

The GuideLLM adapter does **not** automatically create MLflow runs — it returns metrics to the EvalHub API but skips the MLflow logging step (unlike the Korean MCQ adapter which uses `mlflow.log_metrics()` internally).

To keep all results visible in a single MLflow experiment, we manually log the GuideLLM metrics here.

In [12]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Find the GuideLLM completed job from EvalHub
guidellm_metrics = None
guidellm_job_name = None
for j_item in client.jobs.list():
    if j_item.effective_state == JobStatus.COMPLETED and j_item.results:
        for bm in j_item.results.benchmarks:
            if bm.id in ("throughput", "constant", "sweep", "quick_perf_test") and bm.mlflow_run_id is None:
                guidellm_metrics = bm.metrics
                guidellm_job_name = j_item.name
                break
    if guidellm_metrics:
        break

if guidellm_metrics:
    try:
        exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
        if exp is None:
            exp_id = mlflow.create_experiment(EXPERIMENT_NAME)
        else:
            exp_id = exp.experiment_id

        with mlflow.start_run(experiment_id=exp_id, run_name=f"guidellm-{guidellm_job_name}"):
            mlflow.set_tag("benchmark", "guidellm-throughput")
            mlflow.set_tag("model", MODEL_NAME)
            mlflow.set_tag("source", "evalhub-manual-log")
            mlflow.log_params({
                "profile": "throughput",
                "max_seconds": 180,
                "max_requests": 50,
                "data": "prompt_tokens=128,output_tokens=64",
            })
            mlflow.log_metrics(guidellm_metrics)
            print(f"GuideLLM metrics logged to MLflow experiment: {EXPERIMENT_NAME}")
            print(f"  Run name: guidellm-{guidellm_job_name}")
            print(f"  Metrics: {guidellm_metrics}")
    except Exception as e:
        print(f"MLflow logging skipped (not reachable from local): {e}")
        print(f"GuideLLM metrics (from EvalHub):")
        for k, v in sorted(guidellm_metrics.items()):
            print(f"  {k}: {v}")
else:
    print("No GuideLLM results found to log. Run Phase B first.")

MLflow logging skipped (not reachable from local): API request to https://mlflow.redhat-ods-applications.svc:8443/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPSConnectionPool(host='mlflow.redhat-ods-applications.svc', port=8443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=qwen36-27b-full-eval-0617-1428 (Caused by NameResolutionError("HTTPSConnection(host='mlflow.redhat-ods-applications.svc', port=8443): Failed to resolve 'mlflow.redhat-ods-applications.svc' ([Errno 8] nodename nor servname provided, or not known)"))
GuideLLM metrics (from EvalHub):
  mean_itl_ms: 85.45799700327365
  mean_ttft_ms: 235.4049610369133
  output_tokens_per_second: 11.402229222897363
  prompt_tokens_per_second: 25.407045203068346
  requests_per_second: 0.17777777777777778


---

## Step 4: Results Comparison

Compare results across multiple evaluation jobs. Collect metrics from completed jobs and display as a comparison table.

In [13]:
def collect_results_table(client, job_ids=None):
    """Collect benchmark metrics from multiple jobs into a comparison dict.

    Returns: {benchmark_id: {job_name: {metric: value}}}
    """
    if job_ids is None:
        jobs_list = client.jobs.list()
        jobs = [
            j for j in jobs_list
            if j.effective_state == JobStatus.COMPLETED
        ]
    else:
        jobs = [client.jobs.get(jid) for jid in job_ids]

    table = {}
    for j in jobs:
        if not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.id not in table:
                table[bm.id] = {}
            table[bm.id][j.name] = bm.metrics

    return table


comparison = collect_results_table(client)

if comparison:
    print("Results Comparison")
    print("=" * 70)
    for benchmark_id, job_results in comparison.items():
        print(f"\n  Benchmark: {benchmark_id}")
        print(f"  {'-' * 60}")
        for job_name, metrics in job_results.items():
            print(f"    {job_name}:")
            for metric, value in metrics.items():
                if isinstance(value, float):
                    print(f"      {metric:30s} = {value:.4f}")
                else:
                    print(f"      {metric:30s} = {value}")
else:
    print("No completed jobs found. Submit and complete evaluations first.")

Results Comparison

  Benchmark: click
  ------------------------------------------------------------
    unified-click-0617-1428:
      category_accuracy.Economy      = 91.5300
      category_accuracy.Functional   = 89.5200
      category_accuracy.Geography    = 77.7800
      category_accuracy.Grammar      = 56.9000
      category_accuracy.History      = 48.5700
      category_accuracy.Law          = 65.7500
      category_accuracy.Politics     = 85.7100
      category_accuracy.Pop Culture  = 87.8000
      category_accuracy.Society      = 89.9700
      category_accuracy.Textual      = 92.5700
      category_accuracy.Tradition    = 82.8800
      overall_accuracy               = 75.9000
      supercategory_accuracy.Culture = 74.7800
      supercategory_accuracy.Language = 78.3800

  Benchmark: haerae
  ------------------------------------------------------------
    unified-haerae-0617-1428:
      category_accuracy.correct_definition_matching = 0
      category_accuracy.csat_geo     = 2

### Comparison Table with pandas

In [14]:
try:
    import pandas as pd

    rows = []
    for benchmark_id, job_results in comparison.items():
        for job_name, metrics in job_results.items():
            for metric, value in metrics.items():
                if isinstance(value, (int, float)):
                    rows.append({
                        "benchmark": benchmark_id,
                        "job": job_name,
                        "metric": metric,
                        "value": value,
                    })

    if rows:
        df = pd.DataFrame(rows)
        pivot = df.pivot_table(
            index=["benchmark", "metric"],
            columns="job",
            values="value",
        )
        display(pivot.style.format("{:.4f}").highlight_max(axis=1, color="lightgreen"))
    else:
        print("No numeric results to display.")

except ImportError:
    print("pandas not available. Install with: pip install pandas")

---

## Step 5: MLflow Results

EvalHub automatically tracks all evaluation results in MLflow. You do **not** need to install or call `mlflow` directly.

### 1. View Experiment List

Open the MLflow UI in your browser (the URL is printed during EvalHub setup) and navigate to the **Experiments** tab.
Each EvalHub job with an `experiment` parameter creates a corresponding MLflow experiment.

![MLflow Experiment List](../images/eval-result-mlflow.png)

### 2. View Detailed Results & Charts

Click on an experiment to see individual runs. Each run contains:

- **Parameters**: model name, benchmark ID, provider, sample limit
- **Metrics**: `overall_accuracy`, `category_accuracy.*`, `supercategory_accuracy.*`
- **Artifacts**: evaluation logs
- **Traces** (if MLflow Tracing is enabled): individual LLM call prompts/responses

Use the MLflow **Chart** view to compare metrics across runs:
1. Select multiple runs in the experiment
2. Click the **Chart** tab
3. Choose metrics (e.g. `overall_accuracy`) to visualize as bar charts or scatter plots

> **Tip**: The MLflow run ID for each benchmark is available via `job.results.benchmarks[].mlflow_run_id` from the EvalHub SDK.

---

## Step 6: Export Results

### Export to Markdown

In [15]:
def results_to_markdown(comparison, title="EvalHub Benchmark Results"):
    """Convert comparison results to markdown table."""
    if not comparison:
        return "No results available."

    lines = [f"## {title}", ""]

    for benchmark_id, job_results in comparison.items():
        lines.append(f"### {benchmark_id}")
        lines.append("")

        all_metrics = set()
        for metrics in job_results.values():
            all_metrics.update(metrics.keys())
        all_metrics = sorted(all_metrics)

        job_names = sorted(job_results.keys())
        header = "| Metric | " + " | ".join(job_names) + " |"
        sep = "|---" + "|---" * len(job_names) + "|"
        lines.extend([header, sep])

        for metric in all_metrics:
            row = f"| {metric} |"
            for job_name in job_names:
                val = job_results.get(job_name, {}).get(metric, "-")
                if isinstance(val, float):
                    row += f" {val:.4f} |"
                else:
                    row += f" {val} |"
            lines.append(row)

        lines.append("")

    return "\n".join(lines)


md = results_to_markdown(comparison)
print(md)

## EvalHub Benchmark Results

### click

| Metric | unified-click-0617-1428 |
|---|---|
| category_accuracy.Economy | 91.5300 |
| category_accuracy.Functional | 89.5200 |
| category_accuracy.Geography | 77.7800 |
| category_accuracy.Grammar | 56.9000 |
| category_accuracy.History | 48.5700 |
| category_accuracy.Law | 65.7500 |
| category_accuracy.Politics | 85.7100 |
| category_accuracy.Pop Culture | 87.8000 |
| category_accuracy.Society | 89.9700 |
| category_accuracy.Textual | 92.5700 |
| category_accuracy.Tradition | 82.8800 |
| overall_accuracy | 75.9000 |
| supercategory_accuracy.Culture | 74.7800 |
| supercategory_accuracy.Language | 78.3800 |

### haerae

| Metric | unified-haerae-0617-1428 |
|---|---|
| category_accuracy.correct_definition_matching | 0 |
| category_accuracy.csat_geo | 22.2200 |
| category_accuracy.csat_law | 26 |
| category_accuracy.csat_socio | 22.3400 |
| category_accuracy.date_understanding | 0 |
| category_accuracy.general_knowledge | 23.9100 |
| category_a

### Save Results to JSON

In [16]:
import json
from pathlib import Path

results_root = Path("../results")

jobs_list = client.jobs.list()
saved = 0
for j in jobs_list:
    if j.effective_state != JobStatus.COMPLETED or not j.results:
        continue

    model_name = j.model.name if j.model else "unknown"
    model_dir = results_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    job_data = {
        "job_id": j.id,
        "name": j.name,
        "model": {"url": j.model.url, "name": j.model.name},
        "experiment": j.experiment.name if j.experiment else None,
        "benchmarks": [
            {
                "id": bm.id,
                "provider_id": bm.provider_id,
                "metrics": bm.metrics,
                "mlflow_run_id": bm.mlflow_run_id,
            }
            for bm in j.results.benchmarks
        ],
    }

    filename = f"{j.name}_{j.id[:8]}.json"
    output_path = model_dir / filename
    with open(output_path, "w") as f:
        json.dump(job_data, f, indent=2, default=str)
    print(f"Saved: {output_path}")
    saved += 1

print(f"\n{saved} result(s) saved to {results_root.resolve()}/<model-name>/")

Saved: ../results/qwen36-27b/unified-click-0617-1428_ad5af61a.json
Saved: ../results/qwen36-27b/unified-haerae-0617-1428_9e4dfa43.json
Saved: ../results/qwen36-27b/unified-kmmlu_hard-0617-1428_9da5b370.json
Saved: ../results/qwen36-27b/unified-kmmlu-0617-1428_9638a01c.json
Saved: ../results/qwen36-27b/unified-kobest_boolq-0617-1428_28b4c90f.json
Saved: ../results/qwen36-27b/unified-perf-qwen36-27b-0617-1428_15d4b0b0.json

6 result(s) saved to /Users/hyochoi/dev/rhoai-lmeval-builder-lab/results/<model-name>/


---

## Summary

This notebook demonstrated advanced EvalHub evaluation workflows:

1. **Multi-benchmark** evaluation in a single request
2. **Sample size comparison** tracked under one MLflow experiment
3. **Unified evaluation** -- Korean MCQ accuracy + GuideLLM performance in one experiment
4. **Results comparison** with pandas DataFrames

5. **Export** -- Markdown and JSON output

### Next Steps

- **2_eval_hub_kmcq_benchmark/2_summarize_results.ipynb** -- Aggregate and generate reports
- **1_eval_hub_guidellm_benchmark/1_guidellm_benchmark.ipynb** -- Standalone GuideLLM profiling
